In [11]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../../BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [12]:
# LMS Monthly PV / UU

sql = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        EXTRACT(MONTH FROM `Time`) AS _month,
        REGEXP_EXTRACT(EventLabel, r'^(.+?[.]tab)') AS _EventLabel,
        DeviceId
      FROM `openrice-production.ORGA.SV_%s`
      WHERE platform IN ('android', 'ios')
        AND region = 'HK'
        AND (
          EventLabel LIKE '%%HK.LMS2.21.tab%%'
          OR EventLabel LIKE '%%HK.LMS2.9313.tab%%'
        )
    ''', FORMAT_DATE('%Y%m%d', d)),
    ' UNION ALL '
  )
  FROM UNNEST(
    GENERATE_DATE_ARRAY(DATE '2026-09-01', DATE '2026-09-30')
  ) AS d
);

EXECUTE IMMEDIATE FORMAT('''
  SELECT
    _month,
    _EventLabel,
    COUNT(*) AS cnt,
    COUNT(DISTINCT DeviceId) AS device_cnt
  FROM (%s)
  GROUP BY _month, _EventLabel
  ORDER BY _month, _EventLabel
''', sql);
"""

In [13]:
df_lms_monthly = client.query(sql).to_dataframe()
df_lms_monthly

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,_month,_EventLabel,cnt,device_cnt
0,9,HK.LMS2.21.tab,53567,14320
1,9,HK.LMS2.9313.tab,41152,22684


In [20]:
# No. of Search Landmark (nsl): quick + adv

sql_1 = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        deviceid,
        LOWER(eventlabel) AS screen,
        LOWER(LAG(eventlabel) OVER(ORDER BY sessionid, time)) AS prev_screen
      FROM `openrice-production.ORGA.SV_%s`
      WHERE (LOWER(eventlabel) LIKE 'hk.lms2.%%.tab.-%%'
        OR LOWER(eventlabel) = 'hk.qcksearch'
        OR LOWER(eventlabel) LIKE 'hk.advsearch')
    ''', FORMAT_DATE('%Y%m%d', d))
  , ' UNION ALL ')
  FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01','2026-09-30')) d
);

EXECUTE IMMEDIATE
FORMAT(
'''
WITH temp AS (
  %s
)

SELECT
  SUM(CASE WHEN prev_screen = 'hk.qcksearch' THEN 1 END) AS quick_,
  SUM(CASE WHEN prev_screen = 'hk.advsearch' THEN 1 END) AS adv_,
  COUNT(DISTINCT deviceid) AS uu,
  screen
FROM temp
WHERE (screen = 'hk.lms2.21.tab.-990.1'
   OR screen = 'hk.lms2.9313.tab.-990.1')
  AND prev_screen LIKE '%%search%%'
GROUP BY screen
ORDER BY adv_ DESC
''',
sql
);
"""


In [ ]:
df_nsl = client.query(sql_1).to_dataframe()
df_nsl

# No. of Search Landmark (nsl): quick + adv

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,quick_,adv_,uu,screen
0,22501,1482,22296,hk.lms2.9313.tab.-990.1
1,15653,955,14224,hk.lms2.21.tab.-990.1


In [16]:
# LMS Restaurants Total Impressions / Total clicks by LMS (LMS_rest)

sql_2 = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        EventAction,
        event.item.Value AS value
      FROM `openrice-production.ORGA.PV_%s`,
      UNNEST(EventLabel.list) AS event
      WHERE EventAction IN ('impression.poi', 'or.poi.get-details')
        AND event.item.Param = 'sn'
        AND event.item.Value IN ('HK.LMS2.21.tab.-990.1','HK.LMS2.9313.tab.-990.1')
    ''', FORMAT_DATE('%Y%m%d', d))
  , ' UNION ALL ')
  FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01','2026-09-30')) d
);

EXECUTE IMMEDIATE
FORMAT(
'''
SELECT
  EventAction,
  value,
  COUNT(1) AS cnt
FROM (
  %s
)
GROUP BY EventAction, value
''',
sql
);
"""


In [17]:
df_lms_rest = client.query(sql_2).to_dataframe()
df_lms_rest

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,EventAction,value,cnt
0,impression.poi,HK.LMS2.21.tab.-990.1,120744
1,or.poi.get-details,HK.LMS2.21.tab.-990.1,8174
2,or.poi.get-details,HK.LMS2.9313.tab.-990.1,12046
3,impression.poi,HK.LMS2.9313.tab.-990.1,116079


In [22]:
# Top 5 Search Restaurant / Top 5 Tenant Viewed (top5)

sql_3 = r"""
DECLARE query STRING;

## Build per-day UNION ALL query
SET query = (
  WITH all_dates AS (
    SELECT
      EXTRACT(YEAR FROM dt)  AS year,
      EXTRACT(MONTH FROM dt) AS month,
      EXTRACT(DAY FROM dt)   AS day
    FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01', '2026-09-30')) AS dt
  )
  SELECT STRING_AGG(subquery, ' UNION ALL ')
  FROM (
    SELECT FORMAT(
      '''
      SELECT
        SAFE_CAST(event.value.string_value AS INT64) AS poiid
      FROM `openrice-production.analytics_153356786.cluster_events_%04d%02d%02d`,
           UNNEST(event_params) AS event
      WHERE
         user_pseudo_id LIKE '1%%%%'
        AND event_name = 'or_poi_get_details'
        AND event.key = 'POIID'
      ''',
      year, month, day
    ) AS subquery
    FROM all_dates
  )
);

EXECUTE IMMEDIATE FORMAT(
  ''' 
  SELECT
  lp.landmarkid,
  p.poitypeid,
    t.poiid, p.namelang1, 
    COUNT(1) AS cnt
  FROM (
    %s
  ) t
  LEFT JOIN `openrice-production.openrice3.poi` p
    ON t.poiid = p.poiid
  LEFT JOIN `openrice-production.openrice3.landmarkpoi` lp
    ON lp.poiid = p.poiid
  WHERE
    p.regionid = 0
    AND lp.landmarkid in (21,9313)
    AND p.status=10
  GROUP BY lp.landmarkid, p.poitypeid, t.poiid, p.namelang1
  ORDER BY lp.landmarkid, p.poitypeid, cnt desc
  ''',
  query
);
"""

In [ ]:
df_top5 = client.query(sql_3).to_dataframe()

# 顯示 DataFrame 所有列與欄，避免前五／後五縮略
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

df_top5

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
# Banner View IMP (banner_view) - Official Account (featureitem id)

sql_4 = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        event.item.Value AS value
      FROM `openrice-production.ORGA.PV_%s`,
      UNNEST(EventLabel.list) AS event
      WHERE EventAction = 'impression.ad.1'
        AND event.item.Param = 'FeatureItemID'
        AND SAFE_CAST(event.item.Value AS INT64) IN (43016,42986,41971,43017,43081)          -- FeatureItemID
    ''', FORMAT_DATE('%Y%m%d', d))
  , ' UNION ALL ')
  FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01','2026-09-30')) d
);

EXECUTE IMMEDIATE
FORMAT('''
SELECT
  value,
  COUNT(1) AS cnt
FROM (
  %s
)
GROUP BY value
''', sql);

"""


In [ ]:
df_banner_view = client.query(sql_4).to_dataframe()
df_banner_view

# 恆隆66：43016
# Fashion Walk: 42986
# 睇show：41971
# Summer PAWsome Treats: 43017
# 花燈賞：43081

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,value,cnt
0,43017,6918405
1,41971,6090968
2,43081,3329028
3,43016,7377910
4,42986,4959925


In [57]:
# TO GET Banner click id

# import pymssql
# import pandas as pd
# DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}
# with pymssql.connect(DB_info['server'], DB_info['user'], DB_info['password']) as conn:
#     cursor = conn.cursor()
#     sql_clickid = '''
#     use openrice3;
#     SELECT * from featureitem WHERE ItemId in (21,9313)
#     order by 1 desc
#     '''

#     df_clickid = pd.read_sql(sql_clickid, conn)
# df_clickid.to_excel('clickid.xlsx', index=False)

In [ ]:
# Banner Click (banner_click) - Official Account (click id from url)

sql_5 = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        REGEXP_EXTRACT(event.item.Value, r'clk\\?id=(\\d+)') AS clk_id
      FROM `openrice-production.ORGA.PV_%s`,
      UNNEST(EventLabel.list) AS event
      WHERE EventAction = 'or.SR1.ad.1.URL'
        AND event.item.Param = 'url'
        AND REGEXP_CONTAINS(event.item.Value, r'clk\\?id=\\d+')
        AND SAFE_CAST(REGEXP_EXTRACT(event.item.Value, r'clk\\?id=(\\d+)') AS INT64) IN
        (7418544476,7371461683,7289878736,7418574409,7433721277)                                    -- click id
    ''', FORMAT_DATE('%Y%m%d', d))
  , ' UNION ALL ')
  FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01','2026-09-30')) d
);

EXECUTE IMMEDIATE
FORMAT('''
SELECT
  clk_id,
  COUNT(1) AS cnt
FROM (
  %s
)
GROUP BY clk_id
''', sql);

"""

In [ ]:
df_banner_click = client.query(sql_5).to_dataframe()
df_banner_click

# 恆隆66：7418544476
# Fashion Walk: 7371461683
# 睇show：7289878736
# Summer PAWsome Treats: 7418574409
# 花燈賞：7433721277

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,clk_id,cnt
0,7418574409,4238
1,7371461683,3902
2,7418544476,4523
3,7289878736,3973
4,7433721277,2272


In [ ]:
# Banner Clicks - What's News (CorpAccountCampaignId)
# 點擊banner 11:16:20|| or.campaign.getdetails| LndID:21; CityID:0; CampaignID:218;Lang:zh_TW;Ver:7.21.0; sn:hk.LMS2.21.News.1
# 點擊banner 11:18:55|| or.campaign.getdetails| LndID:9313; CityID:0; CampaignID:217;Lang:zh_TW;Ver:7.21.0; sn:hk.LMS2.9313.News.1

sql_6 = r"""
DECLARE sql STRING;

SET sql = (
  SELECT STRING_AGG(
    FORMAT('''
      SELECT
        event.item.Value AS campaign_id
      FROM `openrice-production.ORGA.PV_%s`,
      UNNEST(EventLabel.list) AS event
      WHERE EventAction = 'or.campaign.getdetails'
        AND event.item.Param = 'CampaignID'
        AND SAFE_CAST(event.item.Value AS INT64) IN (208, 182, 214, 211, 209, 201)         -- campaign IDs
    ''', FORMAT_DATE('%Y%m%d', d)), ' UNION ALL ')
  FROM UNNEST(GENERATE_DATE_ARRAY('2026-09-01','2026-09-30')) d
);

EXECUTE IMMEDIATE FORMAT('''
SELECT
  campaign_id,
  COUNT(1) AS cnt
FROM (%s)
GROUP BY campaign_id
''', sql);

"""

In [51]:
df_banner_click_news = client.query(sql_6).to_dataframe()
df_banner_click_news

# CorpAccountCampaignId

# Amoy Plaza
# 1. 恆隆66: 208
# 2. 睇show玩樂證：182

# Fashion Walk
# 1. Celebrate Mid-Autumn: 214
# 2. Summer PAWsome Treats: 211
# 3. 恆隆66: 209
# 4. Fashion Walk Food Street: 201


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,campaign_id,cnt
0,182,7
1,201,11
2,214,4
3,208,5
4,209,15
5,211,7


In [ ]:
# Booking Section refers to Hang_Lung_Monthly_LMS_Booking.sql 0915